In [1]:
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.tools import tool
from rich import print
import os

/Users/diego/.pyenv/versions/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"]="https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]="pr-trustworthy-snail-12"

In [3]:
llm = init_chat_model("gemini-flash-latest", model_provider="google_genai")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
embd = embeddings.embed_query("hola")
print(len(embd))

3072

In [10]:
#llm.invoke("hola soy un pibardo buscando empleo")
@tool(description="Get the current weather in a given location")
def get_weather(location: str) -> str:
    return f"Chove moito por {location}."

model_with_tools = llm.bind_tools([get_weather])
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Never answer questions about the weather")]
ai_message =model_with_tools.invoke(input=messages)
messages.append(ai_message)


In [11]:
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        # Normalize args and call the appropriate tool
        args = tool_call.get("args", {}) or {}
        name = tool_call.get("name")

        if name == "get_weather":
            # args already should be like {"location": "..."}
            tool_result = get_weather.invoke(args)
        elif name == "search":
            # model may have asked to "search" with {'queries': [...]}
            queries = args.get("queries") or args.get("query") or []
            if isinstance(queries, list) and queries:
                tool_result = get_weather.invoke({"location": queries[0]})
            else:
                tool_result = get_weather.invoke({"location": str(args)})
        else:
            # Unknown tool: create a simple placeholder result
            tool_result = {"output": f"Unhandled tool: {name}", "tool_name": name}

        messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
final_response

AIMessage(content=[], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ca512-2306-7c61-b85a-3b8feca17e92-0', usage_metadata={'input_tokens': 212, 'output_tokens': 0, 'total_tokens': 212, 'input_token_details': {'cache_read': 0}})

In [14]:
messages

[HumanMessage(content="What's boimorto", additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boimorto"}'}, '__gemini_function_call_thought_signatures__': {'df8ccb6e-1361-40c3-bd3e-5fd183a80d73': 'EuICCt8CAb4+9vu5p7KpnFhTuQws6HkuLtnmt0UJJIQ1hLBxtuNrsAPEr499r+jmfLKp672wV5Fu1u+oXkvadFpzl+SF0fraNpjNFN1cTALrRl+iVpV+xBej13sUo9DOlrZc8eO1wTWnG6FdnHPUMdqJSCU0J1rRF0hPvdkqu35aiLuOIfZogfGQgPUPdnXD3VOkzjP+cbQyxp2PPKKybo1liBgvPrCKCOX1MsirEtRw5b+tqvGj2fRKIqmgrQuhhzqfDre1iDU4/vTmIkYODy1YpOvmIXd+cMH/0covtoZKh2LFAgClG6tqL869R+uAwq77XJKWSeuZacdh8xp4VrB8ls65Lm0zve8uUwWejl7SixcMYJOzauE4sXfEqs4TFknHP78SQ+ATxZCh9uB6HBr8c/t3CSAjEK7Xp4kGGlU5EqM6p5GVLFkiw1kWvQRu6ca8N64ptq4Dy/qNFcBLZfQ91/0v'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ca4f5-e4c8-7fb0-abb6-cd89d3b836bc-0', tool_calls=[{'name': 'get

In [15]:
messages

[HumanMessage(content="What's boimorto", additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boimorto"}'}, '__gemini_function_call_thought_signatures__': {'df8ccb6e-1361-40c3-bd3e-5fd183a80d73': 'EuICCt8CAb4+9vu5p7KpnFhTuQws6HkuLtnmt0UJJIQ1hLBxtuNrsAPEr499r+jmfLKp672wV5Fu1u+oXkvadFpzl+SF0fraNpjNFN1cTALrRl+iVpV+xBej13sUo9DOlrZc8eO1wTWnG6FdnHPUMdqJSCU0J1rRF0hPvdkqu35aiLuOIfZogfGQgPUPdnXD3VOkzjP+cbQyxp2PPKKybo1liBgvPrCKCOX1MsirEtRw5b+tqvGj2fRKIqmgrQuhhzqfDre1iDU4/vTmIkYODy1YpOvmIXd+cMH/0covtoZKh2LFAgClG6tqL869R+uAwq77XJKWSeuZacdh8xp4VrB8ls65Lm0zve8uUwWejl7SixcMYJOzauE4sXfEqs4TFknHP78SQ+ATxZCh9uB6HBr8c/t3CSAjEK7Xp4kGGlU5EqM6p5GVLFkiw1kWvQRu6ca8N64ptq4Dy/qNFcBLZfQ91/0v'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ca4f5-e4c8-7fb0-abb6-cd89d3b836bc-0', tool_calls=[{'name': 'get

In [8]:
model_with_tools.invoke( messages)

AIMessage(content=[{'type': 'text', 'text': 'Como ben dis, en Boimorto e en boa parte de Galicia a choiva é unha compañeira habitual. Boimorto é un concello da provincia da Coruña, situado no interior, e ten ese clima oceánico tan característico onde as precipitacións son frecuentes, especialmente nesta época do ano.\n\nPara hoxe en **Boimorto**, a previsión indica que seguirá o tempo inestable:\n*   **Estado:** Chuvia moderada.\n*   **Temperatura:** Arredor dos **11°C**.\n*   **Humidade:** Moi alta (preto do 94%), o que aumenta esa sensación de frío e humidade característica do interior galego.\n\nÉ un día perfecto para quedar a cuberto ou, se tes que saír, levar un bo paraugas e botas. Como dicimos por aquí, "se chove, que chova!". 🌧️\n\n¿Necesitas saber algo máis sobre o tempo ou sobre o concello?', 'extras': {'signature': 'Es8GCswGAb4+9vuipwce7sBYd20fXYSMFOR+7CKJXQoWdg2obPcGd9xKGr27jW585g2MOn8HwuAiEni8AfWewQNlHd1HTgcrvydyOexqt5cLw5NmHjo4pa05bJ43ihk2/bSYZRZWEP0ZvJ+VlwCh+JOty5mp6hhir